In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
import statsmodels.api as sm
import seaborn as sns
import bambi as bmb
import arviz as az
import scoringrules as sr

pa_gs = pd.read_csv('data/pa_data.csv')
temps = pd.read_csv("data/pa_temps_supplement.csv")
#combining main csv dataset with supplementary temperature data csv
pa_gs = pa_gs.merge(temps, on=["station_id", "year"], how="left")

#limiting dataset to just time interval of interest
pa_gs = pa_gs[pa_gs['year'] >= 1960]

#splitting dataframe in preparation for lagging of variable data
pa_gs_yr_gsl = pa_gs[['year', 'growing_season_length', 'last_spring_frost_doy', 'first_fall_frost_doy', 'station_id']]
pa_gs_non_gsl = pa_gs.drop(['growing_season_length', 'last_spring_frost_doy', 'first_fall_frost_doy'], axis = 1)

#creating lagged dataset where variable data is paired with GSL
pa_gs_lagged = pa_gs_non_gsl.copy(deep=True)
pa_gs_lagged['year'] += 1
pa_gs_lagged_vars = pa_gs_lagged.merge(pa_gs_yr_gsl, on=["station_id", "year"], how = "left")

In [14]:
def remove_nulls_simple(data_subset, variables):
        # Drop rows where any of the specified variable columns have nulls
        data_subset_cleaned = data_subset.dropna(subset=variables, ignore_index=True)
        # Subsets the cleaned dataframe back into only chosen variables
        vars_cleaned = data_subset_cleaned[variables]
        return vars_cleaned

def list_to_str(list_name):
    string = " + ".join(list_name)
    return string
    
def bayesian_model_unstandardized(data, target_var_name, cov_list):

    cov_str = list_to_str(cov_list)
    cov_list_target_var = list(cov_list) + [target_var_name] + ['year']
    data = remove_nulls_simple(data, cov_list_target_var)
    
    formula = bmb.Formula(f"{target_var_name} ~ {cov_str} + year",
                      f"sigma ~ {cov_str} + year")
    model = bmb.Model(formula, data=data, dropna = True)
    output = model.fit(idata_kwargs={'log_likelihood':True})

    resid = data[target_var_name] - model.predict(output, inplace=False).posterior["mu"].mean(("chain", "draw")).values
    
    return model, output, resid


all_covs_list = ['dtr_annual', 'dtr_spring', 'tmean_spring','tmean_fall','latitude','longitude','tmax_annual','oni_annual',
                'nao_annual','pna_annual','amo_annual','sst_north_atlantic','pwat_station','dewpoint_station',
                'soil_moisture_station','cloud_cover_station','evaporation_station',
                'sst_gulf_mexico','pwat_southeast_us','dewpoint_2m_southeast_us','soil_moisture_southeast_us',
                'cloud_cover_southeast_us','evaporation_southeast_us']

In [15]:
lagged_model, lagged_output, lagged_resid = bayesian_model_unstandardized(pa_gs_lagged_vars, 'growing_season_length', all_covs_list)

                                                            Grad                                                  
  Progress               Draw        Divergen…   Step size   evals       Speed                Elapsed    Remaini…  
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.151       31          128.54 draws/s       0:00:15    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.145       31          104.77 draws/s       0:00:19    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.126       31          117.96 draws/s       0:00:16    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.152       31          116.45 draws/s       0:00:17    0:00:00

/home/reu/.venv/lib/python3.14/site-packages/pymc/sampling/mcmc.py:1163: FutureWarning: Passing `log_likelihood` via `idata_kwargs` is deprecated and will be removed in future versions. Call `pm.compute_log_likelihood(idata)` instead.
  return _sample_return(
Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 25 seconds.


In [16]:
dtr_list = ["dtr_annual"]
lagged_model_dtr_only, lagged_output_dtr_only, lagged_resid_dtr_only = bayesian_model_unstandardized(pa_gs_lagged_vars, 'growing_season_length', dtr_list)

                                                            Grad                                                  
  Progress               Draw        Divergen…   Step size   evals       Speed                Elapsed    Remaini…  
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.981       7           910.07 draws/s       0:00:02    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.860       3           728.57 draws/s       0:00:02    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.830       3           698.26 draws/s       0:00:02    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.926       3           857.70 draws/s       0:00:02    0:00:00

/home/reu/.venv/lib/python3.14/site-packages/pymc/sampling/mcmc.py:1163: FutureWarning: Passing `log_likelihood` via `idata_kwargs` is deprecated and will be removed in future versions. Call `pm.compute_log_likelihood(idata)` instead.
  return _sample_return(
Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 4 seconds.


In [17]:
lagged_lsf_model, lagged_lsf_output, lagged_lsf_resid = bayesian_model_unstandardized(pa_gs_lagged_vars, 'last_spring_frost_doy', all_covs_list)

                                                            Grad                                                  
  Progress               Draw        Divergen…   Step size   evals       Speed                Elapsed    Remaini…  
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.140       31          116.86 draws/s       0:00:17    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.142       31          123.11 draws/s       0:00:16    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.110       31          147.06 draws/s       0:00:13    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.142       31          137.46 draws/s       0:00:14    0:00:00

/home/reu/.venv/lib/python3.14/site-packages/pymc/sampling/mcmc.py:1163: FutureWarning: Passing `log_likelihood` via `idata_kwargs` is deprecated and will be removed in future versions. Call `pm.compute_log_likelihood(idata)` instead.
  return _sample_return(
Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 23 seconds.


In [18]:
lagged_fff_model, lagged_fff_output, lagged_fff_resid = bayesian_model_unstandardized(pa_gs_lagged_vars, 'first_fall_frost_doy', all_covs_list)

                                                            Grad                                                  
  Progress               Draw        Divergen…   Step size   evals       Speed                Elapsed    Remaini…  
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.154       31          120.26 draws/s       0:00:16    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.135       31          151.39 draws/s       0:00:13    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.153       31          114.26 draws/s       0:00:17    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.142       31          149.48 draws/s       0:00:13    0:00:00

/home/reu/.venv/lib/python3.14/site-packages/pymc/sampling/mcmc.py:1163: FutureWarning: Passing `log_likelihood` via `idata_kwargs` is deprecated and will be removed in future versions. Call `pm.compute_log_likelihood(idata)` instead.
  return _sample_return(
Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 23 seconds.


In [19]:
lagged_gsl_predictions = lagged_model.predict(lagged_output, inplace=False).posterior["mu"].mean(("chain", "draw")).values
lagged_gsl_predictions_series = pd.Series(lagged_gsl_predictions)

all_covs_w_gsl = all_covs_list + ['growing_season_length']
pa_gs_lagged_no_na = remove_nulls_simple(pa_gs_lagged_vars, all_covs_w_gsl)
pa_gs_lagged_no_na_gsl = pa_gs_lagged_no_na['growing_season_length']

lagged_gsl_predictions_frame = pd.concat([lagged_gsl_predictions_series, pa_gs_lagged_no_na_gsl, lagged_resid], axis=1)
lagged_gsl_predictions_frame.columns = ['Predicted GSL Value', 'Actual GSL Value', 'Residual Value']

In [20]:
lagged_gsl_dtr_only_predictions = lagged_model_dtr_only.predict(lagged_output_dtr_only, inplace=False).posterior["mu"].mean(("chain", "draw")).values
lagged_gsl_dtr_only_predictions_series = pd.Series(lagged_gsl_dtr_only_predictions)

lagged_gsl_dtr_only_predictions_frame = pd.concat([lagged_gsl_dtr_only_predictions_series, pa_gs_lagged_no_na_gsl, lagged_resid_dtr_only], axis=1)
lagged_gsl_dtr_only_predictions_frame.columns = ['Predicted GSL Value', 'Actual GSL Value', 'Residual Value']

In [21]:
all_covs_w_fff = all_covs_list + ['first_fall_frost_doy']
pa_gs_lagged_no_na_fff = remove_nulls_simple(pa_gs_lagged_vars, all_covs_w_fff)
pa_gs_lagged_no_na_fff = pa_gs_lagged_no_na_fff['first_fall_frost_doy']

all_covs_w_lsf = all_covs_list + ['last_spring_frost_doy']
pa_gs_lagged_no_na_lsf = remove_nulls_simple(pa_gs_lagged_vars, all_covs_w_lsf)
pa_gs_lagged_no_na_lsf = pa_gs_lagged_no_na_lsf['last_spring_frost_doy']

In [22]:
lagged_fff_predictions = lagged_fff_model.predict(lagged_fff_output, inplace=False).posterior["mu"].mean(("chain", "draw")).values
lagged_fff_predictions_series = pd.Series(lagged_fff_predictions)

lagged_fff_predictions_frame = pd.concat([lagged_fff_predictions_series, pa_gs_lagged_no_na_fff, lagged_fff_resid], axis=1)
lagged_fff_predictions_frame.columns = ['Predicted FFF Value', 'Actual FFF Value', 'Residual Value']

In [23]:
lagged_lsf_predictions = lagged_lsf_model.predict(lagged_lsf_output, inplace=False).posterior["mu"].mean(("chain", "draw")).values
lagged_lsf_predictions_series = pd.Series(lagged_lsf_predictions)

lagged_lsf_predictions_frame = pd.concat([lagged_lsf_predictions_series, pa_gs_lagged_no_na_lsf, lagged_lsf_resid], axis=1)
lagged_lsf_predictions_frame.columns = ['Predicted LSF Value', 'Actual LSF Value', 'Residual Value']

In [24]:
lagged_lsf_predictions_frame.head(15)

,Predicted LSF Value,Actual LSF Value,Residual Value
0,112.425387,88.0,-24.425387
1,111.231991,135.0,23.768009
2,118.261569,115.0,-3.261569
3,119.884086,93.0,-26.884086
4,110.689191,105.0,-5.689191
5,112.991303,129.0,16.008697
6,114.971796,121.0,6.028204
7,113.110501,120.0,6.889499
8,125.827758,131.0,5.172242
9,120.979795,117.0,-3.979795


In [33]:
lagged_lsf_preds_obs = lagged_lsf_predictions_frame.drop(["Predicted LSF Value"], axis=1)
lagged_lsf_preds_obs = np.array(lagged_lsf_preds_obs)
lagged_lsf_preds_fcts = lagged_lsf_predictions_frame.drop(["Actual LSF Value"], axis=1)
lagged_lsf_preds_fcts = np.array(lagged_lsf_preds_fcts)

In [34]:
lsf_model_variogram_score = sr.variogram_score(lagged_lsf_preds_obs, lagged_lsf_preds_fcts, m_axis = 0, v_axis = 1)

ValueError: Forecasts shape (5944, 2) and observations shape (5944, 2) are not compatible for broadcasting!